# 日本の住所の表記ゆれを正規化して要素に分ける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/02_string/address_normalize.ipynb)

NLP クックブック 02-20 ／ 文字列処理と正規化

「東京都千代田区丸の内1丁目9番1号」「千代田区丸の内一丁目9-1」「東京都千代田区丸ノ内１－９－１」のような住所の表記ゆれをそろえ、都道府県・市区町村・町丁目・番地に分けたい。

In [1]:
!pip install -q kanjize normalize-japanese-addresses jageocoder

In [2]:
# まず辞書を使わない簡易な方法: NFKC → 漢数字を算用数字に → 「丁目・番・号」をハイフンに → 正規表現で分割
# 漢数字の変換はこの章の「漢数字⇔アラビア数字を変換する」と同じ kanjize (MIT) を使う
import re
import unicodedata
from kanjize import kanji2number, number2kanji

KANSUJI = "〇一二三四五六七八九十百千"
DASH = "-‐‑‒–—―−ー－"  # ハイフンに見える文字 (長音「ー」やマイナス「−」も混ざる)

def simple_normalize(addr):
    s = unicodedata.normalize("NFKC", addr).replace(" ", "")
    # 漢数字は「丁目・番・号・の・ハイフン」の直前にあるものだけ変換する (「八王子」「六本木」「一番町」を守る)
    s = re.sub(rf"[{KANSUJI}]+(?=丁目|番(?!町)|号|の[\d{KANSUJI}]|[{DASH}]|$)", lambda m: str(kanji2number(m.group())), s)
    s = re.sub(rf"(?<=\d)[{DASH}](?=\d)", "-", s)
    has_chome = "丁目" in s
    s = re.sub(r"(?<=\d)(丁目|番地|番|の)(?=\d)", "-", s)
    s = re.sub(r"(?<=\d)(丁目|番地|番|号)", "", s)
    m = re.match(r"(東京都|北海道|(?:京都|大阪)府|.{2,3}県)?((?:.+?郡)?.+?[市区町村](?:.+?区)?)?(\D*)([\d-]*)(.*)", s)
    pref, city, town, nums, rest = (g or "" for g in m.groups())
    nums = [n for n in nums.split("-") if n]
    if nums and (has_chome or len(nums) >= 3):  # 丁目と書いていなくても、数字が 3 つなら先頭を丁目とみなす (推測)
        town += number2kanji(int(nums.pop(0))) + "丁目"
    return {"pref": pref, "city": city, "town": town, "addr": "-".join(nums), "rest": rest}

for a in ["東京都千代田区丸の内1丁目9番1号", "千代田区丸の内一丁目9-1", "東京都千代田区丸ノ内１－９－１",
          "東京都港区六本木六丁目10番1号", "東京都八王子市元本郷町三丁目24番1号", "東京都千代田区丸の内一の九の一",
          "東京都千代田区一番町10-1"]:
    print(a, "→", simple_normalize(a))

東京都千代田区丸の内1丁目9番1号 → {'pref': '東京都', 'city': '千代田区', 'town': '丸の内一丁目', 'addr': '9-1', 'rest': ''}
千代田区丸の内一丁目9-1 → {'pref': '', 'city': '千代田区', 'town': '丸の内一丁目', 'addr': '9-1', 'rest': ''}
東京都千代田区丸ノ内１－９－１ → {'pref': '東京都', 'city': '千代田区', 'town': '丸ノ内一丁目', 'addr': '9-1', 'rest': ''}
東京都港区六本木六丁目10番1号 → {'pref': '東京都', 'city': '港区', 'town': '六本木六丁目', 'addr': '10-1', 'rest': ''}
東京都八王子市元本郷町三丁目24番1号 → {'pref': '東京都', 'city': '八王子市', 'town': '元本郷町三丁目', 'addr': '24-1', 'rest': ''}
東京都千代田区丸の内一の九の一 → {'pref': '東京都', 'city': '千代田区', 'town': '丸の内一丁目', 'addr': '9-1', 'rest': ''}
東京都千代田区一番町10-1 → {'pref': '東京都', 'city': '千代田区', 'town': '一番町', 'addr': '10-1', 'rest': ''}


In [3]:
# 簡易な方法が誤る例
for a in ["三重県四日市市諏訪町1-5",                 # 「四日市」で市が終わったと判断する
          "長野県大町市大町3887",                    # 「大町」で町が終わったと判断する
          "京都市中京区寺町通御池上る上本能寺前町488",  # 通り名が町名に混ざる。都道府県も補えない
          "北海道札幌市西区24-2-2-3-3",              # 「二十四軒二条二丁目」の略記は辞書がないと読めない
          "東京都千代田区霞ヶ関1-3-1"]:               # 正しくは「霞が関」だが、それは辞書がないと分からない
    print(a, "→", simple_normalize(a))

三重県四日市市諏訪町1-5 → {'pref': '三重県', 'city': '四日市', 'town': '市諏訪町', 'addr': '1-5', 'rest': ''}
長野県大町市大町3887 → {'pref': '長野県', 'city': '大町', 'town': '市大町', 'addr': '3887', 'rest': ''}
京都市中京区寺町通御池上る上本能寺前町488 → {'pref': '', 'city': '京都市中京区', 'town': '寺町通御池上る上本能寺前町', 'addr': '488', 'rest': ''}
北海道札幌市西区24-2-2-3-3 → {'pref': '北海道', 'city': '札幌市西区', 'town': '二十四丁目', 'addr': '2-2-3-3', 'rest': ''}
東京都千代田区霞ヶ関1-3-1 → {'pref': '東京都', 'city': '千代田区', 'town': '霞ヶ関一丁目', 'addr': '3-1', 'rest': ''}


In [4]:
# 方法 1: normalize-japanese-addresses (Geolonia の JavaScript 版を Python に移植したもの、MIT)
# 町丁目の一覧を Geolonia 住所データの API (https://geolonia.github.io/japanese-addresses/api/ja) から都度取得する
# 住所データ: Geolonia 住所データ (CC BY 4.0。国土交通省の位置参照情報と日本郵便の郵便番号データをもとに作成)
from normalize_japanese_addresses import normalize

ADDRS = ["東京都千代田区丸の内1丁目9番1号", "千代田区丸の内一丁目9-1", "東京都千代田区丸ノ内１－９－１",
         "三重県四日市市諏訪町1-5", "長野県大町市大町3887", "京都市中京区寺町通御池上る上本能寺前町488",
         "北海道札幌市西区24-2-2-3-3", "東京都千代田区霞ヶ関1-3-1", "大阪府大阪市北区梅田3-1-1 グランフロント大阪"]
for a in ADDRS:
    r = normalize(a)
    print(f"level={r['level']} {r['pref']} | {r['city']} | {r['town']} | {r['addr']}   ← {a}")

level=3 東京都 | 千代田区 | 丸の内一丁目 | 9-1   ← 東京都千代田区丸の内1丁目9番1号
level=3 東京都 | 千代田区 | 丸の内一丁目 | 9-1   ← 千代田区丸の内一丁目9-1
level=3 東京都 | 千代田区 | 丸の内一丁目 | 9-1   ← 東京都千代田区丸ノ内１－９－１
level=3 三重県 | 四日市市 | 諏訪町 | 1-5   ← 三重県四日市市諏訪町1-5
level=3 長野県 | 大町市 | 大町 | 3887   ← 長野県大町市大町3887
level=3 京都府 | 京都市中京区 | 上本能寺前町 | 488   ← 京都市中京区寺町通御池上る上本能寺前町488
level=3 北海道 | 札幌市西区 | 二十四軒二条二丁目 | 3-3   ← 北海道札幌市西区24-2-2-3-3
level=3 東京都 | 千代田区 | 霞が関一丁目 | 3-1   ← 東京都千代田区霞ヶ関1-3-1
level=3 大阪府 | 大阪市北区 | 梅田三丁目 | 1-1 グランフロント大阪   ← 大阪府大阪市北区梅田3-1-1 グランフロント大阪


In [5]:
# 方法 2: jageocoder (東京大学 CSIS のアドレスマッチングで使われている DAMS の Python 移植、MIT)
# 住所辞書は別に入れる。ここでは街区レベルの全国版 (zip で 351MB、展開後 1.5GB 程度) を使う
# 辞書: 株式会社情報試作室「jageocoder 用住所データベース (街区レベル)」2025-04-23 版
#   国土交通省「位置参照情報」(令和5年)、Geolonia 住所データ、デジタル庁「アドレス・ベース・レジストリ」などを加工したもの
#   商用・非商用とも利用可。複製するときは同梱の README.md を同じ場所に置き、成果物を公開するときは出典を書く (README.md の利用条件)
# 2026-04-17 版 (20260417/v2/) もあるが、README と latest/v2/ の一覧が指す 2025-04-23 版を使う
# Colab で数分程度 (ダウンロードと展開)
import os
import urllib.request
import jageocoder

URL = "https://www.info-proto.com/static/jageocoder/20250423/v2/gaiku_all_20250423_v22.zip"
if not os.path.exists("jageocoder_db"):
    urllib.request.urlretrieve(URL, "dic.zip")
    jageocoder.install_dictionary("dic.zip", db_dir="jageocoder_db", skip_confirmation=True)
    os.remove("dic.zip")
jageocoder.init(db_dir="jageocoder_db")  # 辞書を入れずに公開デモサーバを使うなら jageocoder.init(url="https://jageocoder.info-proto.com/jsonrpc")
print(jageocoder.installed_dictionary_readme("jageocoder_db")[:200])  # 利用条件を必ず読む

jageocoder 用住所データベース利用規約（街区レベル）
2025年4月23日 株式会社情報試作室

1. データ形式について

    このデータは jageocoder 用の住所データベースファイルです。

    ファイル名末尾が "_v1*.zip" (* は数字) のものは v1 系のフォーマットで、
    展開すると "address.db" という名前の SQLite3 ファ


In [6]:
# レベル (1: 都道府県, 2: 郡, 3: 市・特別区, 4: 政令市の区, 5: 大字・町, 6: 字・丁目, 7: 街区・地番) で要素に分ける
def jageo_normalize(addr):
    results = jageocoder.searchNode(addr)
    if not results:
        return None
    res = results[0]  # 候補が複数あるときは同点のものが並ぶが、ここでは先頭だけを使う (都道府県などで絞るとよい)
    parts = {"pref": "", "city": "", "town": "", "addr": ""}
    for node in res.node.get_parent_list():
        key = "pref" if node.level == 1 else "city" if node.level <= 4 else "town" if node.level <= 6 else "addr"
        parts[key] += re.sub(r"(番地|番|号)$", "", node.name) if key == "addr" else node.name
    # 街区レベルの辞書には住居番号 (「1号」) がないので、残りの先頭の数字を足す
    rest = unicodedata.normalize("NFKC", addr[len(res.matched):])
    if parts["addr"] and (m := re.match(r"(\d+)号?", rest)):
        parts["addr"] += "-" + m.group(1)
        rest = rest[m.end():]
    parts["rest"] = rest  # 辞書で照合できなかった残り
    parts["n_cand"] = len(results)
    return parts

for a in ADDRS:
    r = jageo_normalize(a)
    if r is None:
        print("(照合できない)   ←", a)
        continue
    print(f"{r['pref']} | {r['city']} | {r['town']} | {r['addr']} | 残り={r['rest']!r}   ← {a}")

東京都 | 千代田区 | 丸の内一丁目 | 9-1 | 残り=''   ← 東京都千代田区丸の内1丁目9番1号
東京都 | 千代田区 | 丸の内一丁目 | 9-1 | 残り=''   ← 千代田区丸の内一丁目9-1
東京都 | 千代田区 | 丸の内一丁目 | 9-1 | 残り=''   ← 東京都千代田区丸ノ内１－９－１
三重県 | 四日市市 | 諏訪町 | 1-5 | 残り=''   ← 三重県四日市市諏訪町1-5
長野県 | 大町市 | 大町 | 3887 | 残り=''   ← 長野県大町市大町3887
京都府 | 京都市中京区 | 上本能寺前町 | 488 | 残り=''   ← 京都市中京区寺町通御池上る上本能寺前町488
北海道 | 札幌市西区 |  |  | 残り='24-2-2-3-3'   ← 北海道札幌市西区24-2-2-3-3
東京都 | 千代田区 | 霞が関一丁目 | 3-1 | 残り=''   ← 東京都千代田区霞ヶ関1-3-1
大阪府 | 大阪市北区 | 梅田三丁目 | 1-1 | 残り=' グランフロント大阪'   ← 大阪府大阪市北区梅田3-1-1 グランフロント大阪


In [7]:
# 3 つの方法がそろって、あるいはどれかが失敗する例
import pandas as pd

def fmt(d):
    return "|".join(d[k] for k in ["pref", "city", "town", "addr"]) if d else "(なし)"

HARD = ["東京千代田区丸の内1-9-1",      # 「都」の抜け
        "東京都千代田区丸野内1-9-1",    # 誤字
        "東京都千代田区丸の内壱丁目9番1号",  # 「壱」のような改ざん防止用の数字
        "東京都千代田区丸の内9丁目1番",  # 存在しない丁目
        "埼玉県浦和市高砂3-15-1",       # 合併前の市名 (現在はさいたま市浦和区)
        "北区梅田3-1-1",                # 都道府県なしの「北区」
        "中央区本町1-1-1"]              # 都道府県なしで、候補が複数ある
rows = []
for a in HARD:
    j, n = jageo_normalize(a), normalize(a)
    rows.append({"入力": a, "簡易": fmt(simple_normalize(a)), "NJA": f"{fmt(n)} (level={n['level']})",
                 "jageocoder": fmt(j) + (f" 残り={j['rest']}" if j and j["rest"] else "")
                               + (f" (候補{j['n_cand']})" if j and j["n_cand"] > 1 else "")})
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,入力,簡易,NJA,jageocoder
0,東京千代田区丸の内1-9-1,|東京千代田区|丸の内一丁目|9-1,東京都|千代田区|丸の内一丁目|9-1 (level=3),東京||| 残り=千代田区丸の内1-9-1
1,東京都千代田区丸野内1-9-1,東京都|千代田区|丸野内一丁目|9-1,東京都|千代田区||丸野内1-9-1 (level=2),東京都|千代田区|| 残り=丸野内1-9-1
2,東京都千代田区丸の内壱丁目9番1号,東京都|千代田区|丸の内壱丁目九丁目|1,東京都|千代田区||丸の内壱丁目9番1号 (level=2),東京都|千代田区|丸の内| 残り=壱丁目9番1号
3,東京都千代田区丸の内9丁目1番,東京都|千代田区|丸の内九丁目|1,東京都|千代田区||丸の内9丁目1番 (level=2),東京都|千代田区|丸の内| 残り=9丁目1番
4,埼玉県浦和市高砂3-15-1,埼玉県|浦和市|高砂三丁目|15-1,埼玉県|||浦和市高砂3-15-1 (level=1),埼玉県|さいたま市浦和区|高砂三丁目|15-1
5,北区梅田3-1-1,|北区|梅田三丁目|1-1,東京都|北区||梅田3-1-1 (level=2),大阪府|大阪市北区|梅田三丁目|1-1
6,中央区本町1-1-1,|中央区|本町一丁目|1-1,東京都|中央区||本町1-1-1 (level=2),千葉県|千葉市中央区|本町一丁目|1-1 (候補2)


- 簡易な方法で揃うのは全角・漢数字・ハイフンまで。市区町村の境目（「四日市」で切れた）や正式な表記（「霞ヶ関」が残った）は辞書がないと決められない。地名の漢数字（「一番町」）は規則で守るので漏れはありうる。
- normalize-japanese-addresses は都道府県を補い（京都府）、通り名を除き、「24-2-2-3-3」を「二十四軒二条二丁目」と読めた。分けるのは町丁目までで、番地と建物名は addr に残る。
- normalize-japanese-addresses は町丁目の一覧を API から取る（住所全体は送らない）。API の最終更新は 2023 年 11 月で（2026 年 11 月に確認）、新しい町名はない可能性がある。
- jageocoder は手元の辞書で照合し、合併前の「浦和市」を「さいたま市浦和区」に直せた（normalize-japanese-addresses は level=1）。「東京千代田区」と「24-2-2-3-3」は照合できなかった。
- 「北区」「中央区」だけだと、normalize-japanese-addresses は東京都と決め打ちし、jageocoder は候補をすべて返す（中央区は 2 つ）。このコードは先頭の候補だけを使うので、都道府県などで絞る。
- 街区レベルの辞書には住居番号（「1号」）がない。住居番号まで入った版 (jukyo_all) は zip で 4.5GB ある。公開デモサーバも使えるが、リクエスト数に上限があり、住所がサーバに送られる。
- 誤字（「丸野内」）、「壱丁目」、存在しない丁目は 3 つとも失敗した。辞書を使う 2 つは町丁目が空か途中で止まるので気づけるが、簡易な方法は「丸の内九丁目」のようなもっともらしい誤りを返す。
- 誤字は、この章の「編集距離とあいまい文字列マッチング (RapidFuzz)」で町丁目の一覧から近いものを探すとよい。文章から住所の範囲を見つけるのは固有表現抽出の章を参照。